***Install Package***

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter


**Catatan**

In [ ]:
# ============================================================
# PARAMETER DINAMIS
# REPORT_DATE akan otomatis mengikuti tanggal maksimum dari
# data transaksi Darkstore yang di-upload.
# REPORT_START otomatis menjadi tanggal 1 pada bulan yang sama.
# Jadi script tidak terkunci pada 19 September.
# ============================================================
REPORT_DATE = None
REPORT_START = None
REPORT_DAYS = None

def set_report_period(max_date):
    global REPORT_DATE, REPORT_START, REPORT_DAYS

    if pd.isna(max_date):
        raise ValueError("Tidak ditemukan tanggal valid pada data transaksi Darkstore.")

    REPORT_DATE = pd.Timestamp(max_date).normalize()
    REPORT_START = REPORT_DATE.replace(day=1)
    REPORT_DAYS = (REPORT_DATE - REPORT_START).days + 1

def period_label():
    return (
        f"1-{REPORT_DATE.day} "
        f"{REPORT_DATE.strftime('%B %Y')}"
    )

def report_date_label():
    return REPORT_DATE.strftime('%d %B %Y').lstrip('0')

def month_sheet_code():
    # Excel reference memakai "SEPT" untuk September.
    # Bulan lain otomatis mengikuti nama singkatnya.
    if REPORT_DATE.month == 9:
        return "SEPT"
    return REPORT_DATE.strftime("%b").upper()

def sheet_names():
    code = month_sheet_code()
    day = REPORT_DATE.day
    return {
        "daily": f"Resume MTD DS {code} DAILY",
        "weekly": f"Resume MTD DS {code} WEEKLY",
        "oos_daily": f"OOS TGL {day}",
        "oos_mtd": f"OOS 1-{day}",
        "sla_daily": f"SLA REMARK {day}",
        "sla_mtd": f"SLA REMARK 1-{day}"
    }

# ============================================================
# HELPER
# ============================================================
def upload_one(title):
    print("=" * 70)
    print(title)
    print("=" * 70)
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError("Harus upload tepat 1 file.")

    filename = next(iter(uploaded))
    print("OK:", filename)
    return filename

def ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = np.where(denominator != 0, numerator / denominator, np.nan)
    if np.ndim(result) == 0:
        return float(result)
    return result

def get_grand_total(df, key_col="Kode Toko"):
    x = df[df[key_col].astype(str).str.upper().eq("GRAND TOTAL")]
    return x.iloc[0] if not x.empty else None

def style_header(ws, row, min_col, max_col, color):
    for c in range(min_col, max_col + 1):
        cell = ws.cell(row, c)
        cell.fill = PatternFill("solid", fgColor=color)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        cell.border = border

def auto_width(ws, max_scan_rows=1000):
    for col_cells in ws.iter_cols(
        min_row=1,
        max_row=min(ws.max_row, max_scan_rows)
    ):
        col_letter = get_column_letter(col_cells[0].column)
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col_cells
        )
        ws.column_dimensions[col_letter].width = min(
            max(max_len + 2, 10), 30
        )


**input file**

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# TAHAP 1 — SLA REMARK / DELIVERY (MENTAH)
# ============================================================
sla_file = upload_one(
    "UPLOAD 1/4 — SLA REMARK / DELIVERY"
)

sla = pd.read_csv(
    sla_file,
    encoding="utf-8-sig"
)

sla.columns = [
    str(c).strip()
    for c in sla.columns
]

sla["TANGGAL"] = pd.to_datetime(
    sla["TANGGAL"],
    errors="coerce"
)

sla["DATE"] = sla["TANGGAL"].dt.normalize()

sla_numeric = [
    "JUMLAH_DELIVERY",
    "DELIVERY_ONTIME",
    "DELIVERY_LATE",
    "DELIVERY_NO_SLA",
    "SALES_ONTIME",
    "SALES_LATE",
    "SALES_NO_SLA",
    "TOTAL_SLA_PREPARATION",
    "TOTAL_SLA_DELIVERY"
]

for c in sla_numeric:
    if c in sla.columns:
        sla[c] = pd.to_numeric(
            sla[c],
            errors="coerce"
        ).fillna(0)

sla = sla.loc[
    sla["NAMA_STORE"].astype(str).str.upper().str.startswith("DS")
].copy()

print("Store SLA Darkstore :", sla["KD_STORE"].nunique())
print("Baris SLA Darkstore :", len(sla))

# ============================================================
# TAHAP 2 — DATA TRANSAKSI MENTAH
# ============================================================
transaction_file = upload_one(
    "UPLOAD 2/4 — DATA TRANSAKSI MENTAH / DETAIL DATA"
)

tx = pd.read_csv(
    transaction_file,
    encoding="utf-8-sig"
)

tx.columns = [
    str(c).replace("\ufeff", "").strip()
    for c in tx.columns
]

# --- FIX: Print columns to identify the correct date column ---
print(f"\nColumns in 'tx' DataFrame after cleaning: {tx.columns}\n")

# Assuming the actual date column is named 'YOUR_ACTUAL_DATE_COLUMN_NAME',
# replace 'TANGGAL' with the correct column name.
# For example, if the column is 'DateTransaction', change 'TANGGAL' to 'DateTransaction'.
# If you want to rename it to 'TANGGAL' for consistency, you can do:
# tx = tx.rename(columns={'YOUR_ACTUAL_DATE_COLUMN_NAME': 'TANGGAL'})
tx["TANGGAL"] = pd.to_datetime(
    tx["TANGGAL"],
    errors="coerce"
)

numeric_columns = [
    "JHK", "SALES", "STRUK", "GM",
    "PERCENT_GM", "SPD", "STD", "APC",
    "JHK_TAGI", "SALES_TAGI", "STRUK_TAGI",
    "GM_TAGI"
]

for c in numeric_columns:
    if c in tx.columns:
        tx[c] = pd.to_numeric(
            tx[c],
            errors="coerce"
        ).fillna(0)

# Ambil hanya Darkstore
tx = tx.loc[
    tx["REMARK"].astype(str).str.upper().eq("DARKSTORE")
].copy()

tx["DATE"] = tx["TANGGAL"].dt.normalize()

# ============================================================
# DETEKSI PERIODE OTOMATIS
# Menggunakan tanggal maksimum dari data transaksi Darkstore.
# Contoh data sekarang: maksimum 19-09-2026 -> MTD 1-19 Sep 2026.
# Jika nanti data maksimum berubah, seluruh report ikut berubah otomatis.
# ============================================================
max_tx_date = tx["DATE"].max()
set_report_period(max_tx_date)

print("Store Darkstore      :", tx["KD_STORE"].nunique())
print("Baris Darkstore      :", len(tx))
print("Tanggal maksimum     :", report_date_label())
print("Periode MTD          :", period_label())
print("Jumlah hari kalender :", REPORT_DAYS)

# ============================================================
# TAHAP 3 — OOS JHK MTD
# ============================================================
oos_mtd_file = upload_one(
    f"UPLOAD 3/4 — OOS JHK MTD / {period_label().upper()}"
)

oos_mtd = pd.read_excel(
    oos_mtd_file
)

oos_mtd.columns = [
    str(c).strip()
    for c in oos_mtd.columns
]

for c in [
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% OOS ALL"
]:
    oos_mtd[c] = pd.to_numeric(
        oos_mtd[c],
        errors="coerce"
    )

# ============================================================
# TAHAP 4 — OOS DAILY
# ============================================================
oos_daily_file = upload_one(
    f"UPLOAD 4/4 — OOS DAILY / {report_date_label().upper()}"
)

oos_daily = pd.read_excel(
    oos_daily_file
)

oos_daily.columns = [
    str(c).strip()
    for c in oos_daily.columns
]

for c in [
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% OOS ALL"
]:
    oos_daily[c] = pd.to_numeric(
        oos_daily[c],
        errors="coerce"
    )

UPLOAD 1/4 — SLA REMARK / DELIVERY


Saving Detail_Data_2026-09-01_2026-09-30 SLA.csv to Detail_Data_2026-09-01_2026-09-30 SLA.csv
OK: Detail_Data_2026-09-01_2026-09-30 SLA.csv
Store SLA Darkstore : 68
Baris SLA Darkstore : 1949
UPLOAD 2/4 — DATA TRANSAKSI MENTAH / DETAIL DATA


Saving Detail_Data_2026-09-01_2026-09-30 TRX.csv to Detail_Data_2026-09-01_2026-09-30 TRX.csv
OK: Detail_Data_2026-09-01_2026-09-30 TRX.csv

Columns in 'tx' DataFrame after cleaning: Index(['TANGGAL', 'KD_STORE', 'NAMA_STORE', 'KD_BRANCH', 'NAMA_BRANCH',
       'REMARK', 'BULAN', 'TAHUN', 'PCT_OOS_OFMB', 'JHK', 'SALES', 'STRUK',
       'GM', 'PERCENT_GM', 'SPD', 'STD', 'APC', 'SHOPPER', 'JHK_PICKUP',
       'SALES_PICKUP', 'GM_PICKUP', 'PERCENT_GM_PICKUP', 'STRUK_PICKUP',
       'SHOPPER_PICKUP', 'JHK_DELIVERY', 'SALES_DELIVERY', 'GM_DELIVERY',
       'PERCENT_GM_DELIVERY', 'STRUK_DELIVERY', 'SHOPPER_DELIVERY',
       'JHK_SELF_SERVICE', 'SALES_SELF_SERVICE', 'GM_SELF_SERVICE',
       'PERCENT_GM_SELF_SERVICE', 'STRUK_SELF_SERVICE', 'SHOPPER_SELF_SERVICE',
       'JHK_TAGI', 'SALES_TAGI', 'STRUK_TAGI', 'TAGI_PCT', 'OTHER_TAGI_PCT',
       'GM_TAGI', 'PERCENT_GM_TAGI', 'SHOPPER_TAGI', 'JHK_DSHUB',
       'SALES_DSHUB', 'GM_DSHUB', 'PERCENT_GM_DSHUB', 'STRUK_DSHUB',
       'SHOPPER_DSHUB

Saving [OOS] By Toko 1-25.xlsx to [OOS] By Toko 1-25.xlsx
OK: [OOS] By Toko 1-25.xlsx
UPLOAD 4/4 — OOS DAILY / 29 SEPTEMBER 2026


Saving [OOS] By Toko 25.xlsx to [OOS] By Toko 25.xlsx
OK: [OOS] By Toko 25.xlsx


**HITUNG SLA**

In [ ]:
# ============================================================
# TAHAP 5 — SLA DAILY
# ============================================================
sla_daily = (
    sla.loc[sla["DATE"].eq(REPORT_DATE)]
    .groupby("KD_STORE", as_index=False)
    .agg(
        DELIVERY_ONTIME=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE=("DELIVERY_LATE", "sum"),
        JUMLAH_DELIVERY=("JUMLAH_DELIVERY", "sum")
    )
)

sla_daily["% Ontime"] = ratio(
    sla_daily["DELIVERY_ONTIME"],
    sla_daily["JUMLAH_DELIVERY"]
)

sla_daily["% Late"] = ratio(
    sla_daily["DELIVERY_LATE"],
    sla_daily["JUMLAH_DELIVERY"]
)

# ============================================================
# TAHAP 6 — SLA HARIAN SEMUA HARI
# Dipakai untuk AVG ONTIME/LATE per darkstore
# ============================================================
sla_daily_all = (
    sla.groupby(
        ["KD_STORE", "DATE"],
        as_index=False
    )
    .agg(
        DELIVERY_ONTIME=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE=("DELIVERY_LATE", "sum"),
        JUMLAH_DELIVERY=("JUMLAH_DELIVERY", "sum")
    )
)

sla_daily_all["% Ontime"] = ratio(
    sla_daily_all["DELIVERY_ONTIME"],
    sla_daily_all["JUMLAH_DELIVERY"]
)

sla_daily_all["% Late"] = ratio(
    sla_daily_all["DELIVERY_LATE"],
    sla_daily_all["JUMLAH_DELIVERY"]
)

# PENTING: untuk MTD, % Ontime/% Late bukan rata-rata persentase harian.
# Harus dihitung secara weighted dari total delivery MTD:
#   % Ontime = total DELIVERY_ONTIME / total JUMLAH_DELIVERY
#   % Late   = total DELIVERY_LATE   / total JUMLAH_DELIVERY
# Ini mengikuti perhitungan Excel QUARTILE.INC pada Resume asli.
sla_mtd_metric = (
    sla.loc[
        sla["DATE"].between(REPORT_START, REPORT_DATE)
    ]
    .groupby("KD_STORE", as_index=False)
    .agg(
        DELIVERY_ONTIME_MTD=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE_MTD=("DELIVERY_LATE", "sum"),
        JUMLAH_DELIVERY_MTD=("JUMLAH_DELIVERY", "sum")
    )
)

sla_mtd_metric["% Ontime"] = ratio(
    sla_mtd_metric["DELIVERY_ONTIME_MTD"],
    sla_mtd_metric["JUMLAH_DELIVERY_MTD"]
)

sla_mtd_metric["% Late"] = ratio(
    sla_mtd_metric["DELIVERY_LATE_MTD"],
    sla_mtd_metric["JUMLAH_DELIVERY_MTD"]
)


**PERFOMANCE DAILY**

In [ ]:
sla_daily_all = (
    sla.groupby(
        ["KD_STORE", "DATE"],
        as_index=False
    )
    .agg(
        DELIVERY_ONTIME=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE=("DELIVERY_LATE", "sum"),
        JUMLAH_DELIVERY=("JUMLAH_DELIVERY", "sum")
    )
)

sla_daily_all["% Ontime"] = ratio(
    sla_daily_all["DELIVERY_ONTIME"],
    sla_daily_all["JUMLAH_DELIVERY"]
)

sla_daily_all["% Late"] = ratio(
    sla_daily_all["DELIVERY_LATE"],
    sla_daily_all["JUMLAH_DELIVERY"]
)

# PENTING: untuk MTD, % Ontime/% Late bukan rata-rata persentase harian.
# Harus dihitung secara weighted dari total delivery MTD:
#   % Ontime = total DELIVERY_ONTIME / total JUMLAH_DELIVERY
#   % Late   = total DELIVERY_LATE   / total JUMLAH_DELIVERY
# Ini mengikuti perhitungan Excel QUARTILE.INC pada Resume asli.
sla_mtd_metric = (
    sla.loc[
        sla["DATE"].between(REPORT_START, REPORT_DATE)
    ]
    .groupby("KD_STORE", as_index=False)
    .agg(
        DELIVERY_ONTIME_MTD=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE_MTD=("DELIVERY_LATE", "sum"),
        JUMLAH_DELIVERY_MTD=("JUMLAH_DELIVERY", "sum")
    )
)

sla_mtd_metric["% Ontime"] = ratio(
    sla_mtd_metric["DELIVERY_ONTIME_MTD"],
    sla_mtd_metric["JUMLAH_DELIVERY_MTD"]
)

sla_mtd_metric["% Late"] = ratio(
    sla_mtd_metric["DELIVERY_LATE_MTD"],
    sla_mtd_metric["JUMLAH_DELIVERY_MTD"]
)

**PERFOMANCE DARKSTORE MTD**

In [ ]:
tx_daily = tx.loc[
    tx["DATE"].eq(REPORT_DATE)
].copy()

daily = (
    tx_daily
    .groupby(
        ["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"],
        as_index=False
    )
    .agg(
        JHK=("JHK", "sum"),
        SALES=("SALES", "sum"),
        SALES_TAGI=("SALES_TAGI", "sum"),
        JHK_TAGI=("JHK_TAGI", "sum"),
        STRUK=("STRUK", "sum"),
        GM=("GM", "sum")
    )
)

daily["SPD"] = ratio(
    daily["SALES"],
    daily["JHK"]
)

daily["SPD TAG I"] = ratio(
    daily["SALES_TAGI"],
    daily["JHK_TAGI"]
)

daily["STD"] = ratio(
    daily["STRUK"],
    daily["JHK"]
)

daily["APC"] = ratio(
    daily["SALES"],
    daily["STRUK"]
)

daily["%GM"] = ratio(
    daily["GM"],
    daily["SALES"]
)

daily = daily.merge(
    oos_daily[
        [
            "Kode Toko",
            "% OOS TAG I",
            "% OOS OFMB",
            "% OOS TAG K"
        ]
    ],
    left_on="KD_STORE",
    right_on="Kode Toko",
    how="left"
).drop(columns=["Kode Toko"])

daily = daily.merge(
    sla_daily[
        ["KD_STORE", "% Ontime", "% Late"]
    ],
    on="KD_STORE",
    how="left"
)
tx_mtd = tx.loc[
    tx["DATE"].between(
        REPORT_START,
        REPORT_DATE
    )
].copy()

mtd_sum = (
    tx_mtd
    .groupby(
        ["KD_STORE", "NAMA_STORE", "NAMA_BRANCH"],
        as_index=False
    )
    .agg(
        SALES=("SALES", "sum"),
        SALES_TAGI=("SALES_TAGI", "sum"),
        STRUK=("STRUK", "sum"),
        GM=("GM", "sum")
    )
)

mtd_avg = (
    tx_mtd
    .groupby("KD_STORE", as_index=False)
    .agg(
        APC=("APC", "mean"),
        PERCENT_GM=("PERCENT_GM", "mean")
    )
)

mtd = mtd_sum.merge(
    mtd_avg,
    on="KD_STORE",
    how="left"
)

# JHK MTD = JUMLAH HARI AKTUAL PER TOKO
# Jika toko hanya mempunyai data 17 hari, maka JHK = 17, bukan 19.
mtd_jhk = (
    tx_mtd
    .groupby("KD_STORE")["DATE"]
    .nunique()
    .reset_index(name="JHK")
)

mtd = mtd.merge(
    mtd_jhk,
    on="KD_STORE",
    how="left"
)

mtd["JHK"] = mtd["JHK"].fillna(0)

# Sales / productivity memakai JHK aktual masing-masing toko
mtd["SPD"] = ratio(
    mtd["SALES"],
    mtd["JHK"]
)

mtd["SPD TAG I"] = ratio(
    mtd["SALES_TAGI"],
    mtd["JHK"]
)

mtd["STD"] = ratio(
    mtd["STRUK"],
    mtd["JHK"]
)

mtd["%GM"] = mtd["PERCENT_GM"]

# OOS langsung dari bahan OOS MTD
mtd = mtd.merge(
    oos_mtd[
        [
            "Kode Toko",
            "% OOS TAG I",
            "% OOS OFMB",
            "% OOS TAG K"
        ]
    ],
    left_on="KD_STORE",
    right_on="Kode Toko",
    how="left"
).drop(columns=["Kode Toko"])

# % Ontime / Late MTD = weighted dari total delivery MTD.
# Ini yang membuat hasil store dan quartile sama dengan Excel asli.
mtd = mtd.merge(
    sla_mtd_metric[
        ["KD_STORE", "% Ontime", "% Late"]
    ],
    on="KD_STORE",
    how="left"
)


**GABUNGAN DAILY DAN MTD DARKSTORE PERFOMANCE**

In [ ]:
final_base = mtd.merge(
    daily[
        [
            "KD_STORE",
            "JHK",
            "SALES",
            "SALES_TAGI",
            "SPD",
            "SPD TAG I",
            "STD",
            "APC",
            "%GM",
            "% OOS TAG I",
            "% OOS OFMB",
            "% OOS TAG K",
            "% Ontime",
            "% Late"
        ]
    ],
    on="KD_STORE",
    how="left",
    suffixes=("_MTD", "_DAILY")
)

final_base = final_base.sort_values(
    ["SPD_DAILY", "KD_STORE"],
    ascending=[False, True]
).reset_index(drop=True)

**RESUME DARSTORE PERFOMANCE DAILY**

In [ ]:
final_report = pd.DataFrame({
    "No": np.arange(
        1,
        len(final_base) + 1
    ),
    "Kode": final_base["KD_STORE"],
    "Toko": final_base["NAMA_STORE"],
    "Cabang": final_base["NAMA_BRANCH"],

    "Daily JHK": final_base["JHK_DAILY"],
    "Daily SPD": final_base["SPD_DAILY"],
    "Daily SPD TAG I": final_base["SPD TAG I_DAILY"],
    "Daily STD": final_base["STD_DAILY"],
    "Daily APC": final_base["APC_DAILY"],
    "Daily %GM": final_base["%GM_DAILY"],
    "Daily % OOS TAG I": final_base["% OOS TAG I_DAILY"],
    "Daily % OOS OFMB": final_base["% OOS OFMB_DAILY"],
    "Daily % OOS TAG K": final_base["% OOS TAG K_DAILY"],
    "Daily % Ontime": final_base["% Ontime_DAILY"],
    "Daily % Late": final_base["% Late_DAILY"],

    "MTD JHK": final_base["JHK_MTD"],
    "Total Nett Sales": final_base["SALES_MTD"],
    "Total Nett Sales TAG I": final_base["SALES_TAGI_MTD"],
    "MTD SPD": final_base["SPD_MTD"],
    "MTD SPD TAG I": final_base["SPD TAG I_MTD"],
    "MTD STD": final_base["STD_MTD"],
    "MTD APC": final_base["APC_MTD"],
    "MTD %GM": final_base["%GM_MTD"],
    "MTD % OOS TAG I": final_base["% OOS TAG I_MTD"],
    "MTD % OOS OFMB": final_base["% OOS OFMB_MTD"],
    "MTD % OOS TAG K": final_base["% OOS TAG K_MTD"],
    "MTD % Ontime": final_base["% Ontime_MTD"],
    "MTD % Late": final_base["% Late_MTD"]
})

# ============================================================
# TAHAP 11 — BOTTOM AVERAGE / SUM
# ============================================================
daily_gt = get_grand_total(oos_daily)
mtd_gt = get_grand_total(oos_mtd)

# Daily bottom
# JHK pada baris bawah = MAX hari, bukan SUM seluruh toko.
daily_bottom = {
    "JHK": daily["JHK"].max(),
    "SPD": daily["SALES"].sum() / daily["JHK"].sum(),
    "SPD TAG I": daily["SALES_TAGI"].sum() / daily["JHK_TAGI"].sum(),
    "STD": daily["STRUK"].sum() / daily["JHK"].sum(),
    "APC": daily["SALES"].sum() / daily["STRUK"].sum(),
    "%GM": daily["%GM"].mean(),
    "% OOS TAG I": (
        float(daily_gt["% OOS TAG I"])
        if daily_gt is not None
        else daily["% OOS TAG I"].mean()
    ),
    "% OOS OFMB": (
        float(daily_gt["% OOS OFMB"])
        if daily_gt is not None
        else daily["% OOS OFMB"].mean()
    ),
    "% OOS TAG K": (
        float(daily_gt["% OOS TAG K"])
        if daily_gt is not None
        else daily["% OOS TAG K"].mean()
    ),
    "% Ontime": daily["% Ontime"].mean(),
    "% Late": daily["% Late"].mean()
}

# MTD bottom
# JHK pada baris bawah = MAX hari aktual per toko.
# Perhitungan aggregate productivity menggunakan total JHK aktual.
mtd_jhk_total = mtd["JHK"].sum()

mtd_bottom = {
    "JHK": mtd["JHK"].max(),
    "Total Nett Sales": mtd["SALES"].sum(),
    "Total Nett Sales TAG I": mtd["SALES_TAGI"].sum(),
    "SPD": ratio(mtd["SALES"].sum(), mtd_jhk_total),
    "SPD TAG I": ratio(mtd["SALES_TAGI"].sum(), mtd_jhk_total),
    "STD": ratio(mtd["STRUK"].sum(), mtd_jhk_total),
    "APC": mtd["APC"].mean(),
    "%GM": mtd["%GM"].mean(),
    "% OOS TAG I": (
        float(mtd_gt["% OOS TAG I"])
        if mtd_gt is not None
        else mtd["% OOS TAG I"].mean()
    ),
    "% OOS OFMB": (
        float(mtd_gt["% OOS OFMB"])
        if mtd_gt is not None
        else mtd["% OOS OFMB"].mean()
    ),
    "% OOS TAG K": (
        float(mtd_gt["% OOS TAG K"])
        if mtd_gt is not None
        else mtd["% OOS TAG K"].mean()
    ),
    "% Ontime": mtd["% Ontime"].mean(),
    "% Late": mtd["% Late"].mean()
}

**PERFORMANCE ANALYSIS WEEKLY**

In [ ]:
# PENTING:
# SPD menjadi kategori UTAMA.
# Baru %Ontime menjadi kategori KEDUA.
# Jadi:
# SPD Top 25%
#   -> OTD Top 25%
#   -> OTD 50-75%
#   -> OTD 25-50%
#   -> OTD Bottom 25%
#
# lalu SPD 50-75%, dst.
# ============================================================
# Resume Analysis menggunakan SALES PERFORMANCE MTD DINAMIS.
# Jadi SPD dan % Ontime yang dipakai berasal dari MTD periode 1 s.d. tanggal maksimum data,
# bukan hanya daily pada tanggal maksimum.
analysis = mtd[
    [
        "KD_STORE",
        "NAMA_STORE",
        "NAMA_BRANCH",
        "SPD",
        "% Ontime"
    ]
].copy()

analysis = analysis.rename(
    columns={
        "SPD": "MTD SPD",
        "% Ontime": "MTD % Ontime"
    }
)

# Excel QUARTILE.INC equivalent: explicit linear interpolation.
q_spd = analysis["MTD SPD"].quantile(
    [0.25, 0.50, 0.75],
    interpolation="linear"
)

q_otd = analysis["MTD % Ontime"].quantile(
    [0.25, 0.50, 0.75],
    interpolation="linear"
)

print("\n========================================")
print("QUARTILE RESUME ANALYSIS — QUARTILE.INC")
print("========================================")
print("Periode :", period_label())
print(f"OTD Q1 : {q_otd.loc[0.25]:.4%}")
print(f"OTD Q2 : {q_otd.loc[0.50]:.4%}")
print(f"OTD Q3 : {q_otd.loc[0.75]:.4%}")
print(f"SPD Q1 : {q_spd.loc[0.25]:,.0f}")
print(f"SPD Q2 : {q_spd.loc[0.50]:,.0f}")
print(f"SPD Q3 : {q_spd.loc[0.75]:,.0f}")

def quartile_category(value, q):
    if pd.isna(value):
        return "No Data"
    if value < q.loc[0.25]:
        return "Bottom 25%"
    elif value < q.loc[0.50]:
        return "25-50%"
    elif value < q.loc[0.75]:
        return "50-75%"
    else:
        return "Top 25%"

analysis["Kategori SPD"] = analysis[
    "MTD SPD"
].apply(
    lambda x: quartile_category(
        x,
        q_spd
    )
)

analysis["Kategori % Ontime"] = analysis[
    "MTD % Ontime"
].apply(
    lambda x: quartile_category(
        x,
        q_otd
    )
)

# SPD category HARUS menjadi prioritas pengurutan
spd_order = {
    "Top 25%": 0,
    "50-75%": 1,
    "25-50%": 2,
    "Bottom 25%": 3
}

otd_order = {
    "Top 25%": 0,
    "50-75%": 1,
    "25-50%": 2,
    "Bottom 25%": 3
}

analysis["_SPD_ORDER"] = analysis[
    "Kategori SPD"
].map(spd_order)

analysis["_OTD_ORDER"] = analysis[
    "Kategori % Ontime"
].map(otd_order)

analysis = (
    analysis
    .sort_values(
        ["_SPD_ORDER", "_OTD_ORDER"],
        kind="stable"
    )
    .drop(
        columns=[
            "_SPD_ORDER",
            "_OTD_ORDER"
        ]
    )
    .reset_index(drop=True)
)


QUARTILE RESUME ANALYSIS — QUARTILE.INC
Periode : 1-29 September 2026
OTD Q1 : 67.8743%
OTD Q2 : 79.4430%
OTD Q3 : 88.8965%
SPD Q1 : 39,798,292
SPD Q2 : 45,988,930
SPD Q3 : 55,924,032


**SHEET PENDUKUNG**

In [ ]:
# ============================================================
# TAHAP 13 — PIVOT DAILY
# ============================================================
pivot_daily = daily[
    [
        "KD_STORE",
        "NAMA_STORE",
        "NAMA_BRANCH",
        "JHK",
        "SPD",
        "SPD TAG I",
        "STD",
        "APC",
        "%GM",
        "% OOS TAG I",
        "% OOS OFMB",
        "% OOS TAG K",
        "% Ontime",
        "% Late"
    ]
].copy()

pivot_daily.columns = [
    "Kode",
    "Toko",
    "Cabang",
    "JHK",
    "SPD",
    "SPD TAG I",
    "STD",
    "APC",
    "Average of PERCENT_GM",
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% Ontime",
    "% Late"
]

pivot_daily = pivot_daily.sort_values(
    "Kode"
).reset_index(drop=True)

# ============================================================
# TAHAP 14 — PIVOT WEEKLY
# ============================================================
pivot_weekly = mtd[
    [
        "KD_STORE",
        "NAMA_STORE",
        "NAMA_BRANCH",
        "JHK",
        "SALES",
        "SALES_TAGI",
        "SPD",
        "SPD TAG I",
        "%GM",
        "% OOS TAG I",
        "% OOS OFMB",
        "% OOS TAG K",
        "% Ontime",
        "% Late"
    ]
].copy()

pivot_weekly.columns = [
    "KD_STORE",
    "NAMA_STORE",
    "NAMA_BRANCH",
    "Sum of JHK",
    "Sum of SPD",
    "Sum of SALES_TAGI",
    "Average of SPD2",
    "Average of SALES_TAGI2",
    "Average of PERCENT_GM",
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% Ontime",
    "% Late"
]

pivot_weekly = pivot_weekly.sort_values(
    "KD_STORE"
).reset_index(drop=True)

# ============================================================
# TAHAP 15 — SLA REMARK TANGGAL MAKSIMUM DATA
# ============================================================
sla_19 = sla_daily.rename(
    columns={
        "KD_STORE": "Row Labels",
        "DELIVERY_ONTIME": "Sum of DELIVERY_ONTIME",
        "DELIVERY_LATE": "Sum of DELIVERY_LATE",
        "JUMLAH_DELIVERY": "Sum of JUMLAH_DELIVERY"
    }
)[
    [
        "Row Labels",
        "Sum of DELIVERY_ONTIME",
        "Sum of DELIVERY_LATE",
        "Sum of JUMLAH_DELIVERY",
        "% Ontime",
        "% Late"
    ]
].sort_values("Row Labels").reset_index(drop=True)

# Add Grand Total
sla_19_gt = pd.DataFrame([{
    "Row Labels": "Grand Total",
    "Sum of DELIVERY_ONTIME": sla_19[
        "Sum of DELIVERY_ONTIME"
    ].sum(),
    "Sum of DELIVERY_LATE": sla_19[
        "Sum of DELIVERY_LATE"
    ].sum(),
    "Sum of JUMLAH_DELIVERY": sla_19[
        "Sum of JUMLAH_DELIVERY"
    ].sum(),
    "% Ontime": sla_19[
        "Sum of DELIVERY_ONTIME"
    ].sum() / sla_19[
        "Sum of JUMLAH_DELIVERY"
    ].sum(),
    "% Late": sla_19[
        "Sum of DELIVERY_LATE"
    ].sum() / sla_19[
        "Sum of JUMLAH_DELIVERY"
    ].sum()
}])

sla_19 = pd.concat(
    [sla_19, sla_19_gt],
    ignore_index=True
)

# ============================================================
# TAHAP 16 — SLA REMARK MTD DINAMIS
# ============================================================
sla_mtd_sum = (
    sla
    .groupby("KD_STORE", as_index=False)
    .agg(
        JUMLAH_DELIVERY=("JUMLAH_DELIVERY", "sum"),
        DELIVERY_ONTIME=("DELIVERY_ONTIME", "sum"),
        DELIVERY_LATE=("DELIVERY_LATE", "sum")
    )
)

sla_mtd_out = sla_mtd_sum.rename(
    columns={
        "KD_STORE": "Kode",
        "JUMLAH_DELIVERY": "Sum of JUMLAH_DELIVERY",
        "DELIVERY_ONTIME": "Sum of DELIVERY_ONTIME",
        "DELIVERY_LATE": "Sum of DELIVERY_LATE"
    }
)

sla_mtd_out["% Ontime"] = (
    sla_mtd_out["Sum of DELIVERY_ONTIME"]
    / sla_mtd_out["Sum of JUMLAH_DELIVERY"]
)

sla_mtd_out["% Late"] = (
    sla_mtd_out["Sum of DELIVERY_LATE"]
    / sla_mtd_out["Sum of JUMLAH_DELIVERY"]
)

sla_mtd_out = sla_mtd_out.sort_values(
    "Kode"
).reset_index(drop=True)

sla_mtd_gt = pd.DataFrame([{
    "Kode": "Grand Total",
    "Sum of JUMLAH_DELIVERY":
        sla_mtd_out["Sum of JUMLAH_DELIVERY"].sum(),
    "Sum of DELIVERY_ONTIME":
        sla_mtd_out["Sum of DELIVERY_ONTIME"].sum(),
    "Sum of DELIVERY_LATE":
        sla_mtd_out["Sum of DELIVERY_LATE"].sum(),
    "% Ontime":
        sla_mtd_out["Sum of DELIVERY_ONTIME"].sum()
        / sla_mtd_out["Sum of JUMLAH_DELIVERY"].sum(),
    "% Late":
        sla_mtd_out["Sum of DELIVERY_LATE"].sum()
        / sla_mtd_out["Sum of JUMLAH_DELIVERY"].sum()
}])

sla_mtd_out = pd.concat(
    [sla_mtd_out, sla_mtd_gt],
    ignore_index=True
)

**RESUME DAILY SHARE**

In [ ]:
BLUE_HEADER = "B7C9D6"
BLUE_LIGHT = "D9EAF2"
GREEN = "C6EFCE"
YELLOW = "FFE699"
ORANGE = "F4B183"
RED = "F4CCCC"

thin = Side(
    style="thin",
    color="7F7F7F"
)

border = Border(
    left=thin,
    right=thin,
    top=thin,
    bottom=thin
)

OUTPUT_FILE = f"Resume MTD DS {REPORT_DATE.day} {REPORT_DATE.strftime('%B %Y').upper()}.xlsx"

wb = Workbook()

# ============================================================
# SHEET 1 — RESUME DAILY
# ============================================================
ws = wb.active
ws.title = sheet_names()["daily"]
ws.sheet_view.showGridLines = False

ws["B1"] = "Daily Report Perfomance Darkstore"
ws["B2"] = period_label()

ws["B1"].font = Font(
    bold=True,
    size=14
)

ws["B2"].font = Font(
    bold=True
)

# Merge header
ws.merge_cells("B3:B4")
ws.merge_cells("C3:C4")
ws.merge_cells("D3:D4")
ws.merge_cells("E3:E4")
ws.merge_cells("F3:P3")
ws.merge_cells("Q3:AC3")

ws["B3"] = "No"
ws["C3"] = "Kode"
ws["D3"] = "Toko"
ws["E3"] = "Cabang"

ws["F3"] = (
    "Sales Perfomance Darkstore "
    f"{report_date_label()}"
)

ws["Q3"] = (
    "Sales Perfomance Darkstore "
    f"{period_label()}"
)

daily_headers = [
    "JHK",
    "SPD",
    "SPD TAG I",
    "STD",
    "APC",
    "%GM",
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% Ontime",
    "% Late"
]

mtd_headers = [
    "JHK",
    "Total Nett Sales",
    "Total Nett Sales TAG I",
    "SPD",
    "SPD TAG I",
    "STD",
    "APC",
    "%GM",
    "% OOS TAG I",
    "% OOS OFMB",
    "% OOS TAG K",
    "% Ontime",
    "% Late"
]

for i, h in enumerate(
    daily_headers,
    start=6
):
    ws.cell(4, i).value = h

for i, h in enumerate(
    mtd_headers,
    start=17
):
    ws.cell(4, i).value = h

style_header(
    ws,
    3,
    2,
    29,
    BLUE_HEADER
)

style_header(
    ws,
    4,
    6,
    29,
    BLUE_HEADER
)

# Data
for idx, row in final_report.iterrows():
    excel_row = 5 + idx

    for col_idx, value in enumerate(
        row.tolist(),
        start=2
    ):
        cell = ws.cell(
            excel_row,
            col_idx
        )
        cell.value = value
        cell.border = border

# Bottom row
bottom_row = 5 + len(final_report)

bottom_values = [
    "Average/ Sum",
    None,
    None,
    None,

    daily_bottom["JHK"],
    daily_bottom["SPD"],
    daily_bottom["SPD TAG I"],
    daily_bottom["STD"],
    daily_bottom["APC"],
    daily_bottom["%GM"],
    daily_bottom["% OOS TAG I"],
    daily_bottom["% OOS OFMB"],
    daily_bottom["% OOS TAG K"],
    daily_bottom["% Ontime"],
    daily_bottom["% Late"],

    mtd_bottom["JHK"],
    mtd_bottom["Total Nett Sales"],
    mtd_bottom["Total Nett Sales TAG I"],
    mtd_bottom["SPD"],
    mtd_bottom["SPD TAG I"],
    mtd_bottom["STD"],
    mtd_bottom["APC"],
    mtd_bottom["%GM"],
    mtd_bottom["% OOS TAG I"],
    mtd_bottom["% OOS OFMB"],
    mtd_bottom["% OOS TAG K"],
    mtd_bottom["% Ontime"],
    mtd_bottom["% Late"]
]

for col_idx, value in enumerate(
    bottom_values,
    start=2
):
    cell = ws.cell(
        bottom_row,
        col_idx
    )
    if isinstance(value, np.ndarray):
        value = float(value.item()) if value.ndim == 0 else value.tolist()
    elif isinstance(value, np.generic):
        value = value.item()
    cell.value = value
    cell.font = Font(
        bold=True
    )
    cell.border = border

ws.merge_cells(
    start_row=bottom_row,
    start_column=2,
    end_row=bottom_row,
    end_column=5
)

# Number formats
# JHK adalah JUMLAH HARI, bukan persentase.
# Daily JHK = 1 untuk 1 hari.
# MTD JHK = jumlah hari aktual per toko.
percent_cols = [
    "K", "L", "M", "N", "O", "P",
    "X", "Y", "Z", "AA", "AB", "AC"
]

number_cols = [
    "F",
    "G", "H", "I", "J",
    "Q", "R", "S", "T", "U",
    "V", "W"
]

for col in percent_cols:
    for r in range(
        5,
        bottom_row + 1
    ):
        ws[f"{col}{r}"].number_format = "0.00%"

for col in number_cols:
    for r in range(
        5,
        bottom_row + 1
    ):
        ws[f"{col}{r}"].number_format = "#,##0"

# ============================================================
# ALIGNMENT + TOP 5 / BOTTOM 5 DAILY
# ============================================================
# Seluruh area data dibuat berkotak dan alignment dibuat konsisten.
# Highlight hanya pada 4 metric berikut:
#   G = SPD
#   H = SPD TAG I
#   O = % Ontime
#   P = % Late
#
# Untuk SPD, SPD TAG I, dan % Ontime:
#   Top 5    = HIJAU
#   Bottom 5 = MERAH
#
# Untuk % Late dibalik karena semakin kecil semakin baik:
#   Late terendah (Bottom 5 nilai) = HIJAU
#   Late tertinggi (Top 5 nilai)   = MERAH
#
# Ranking dilakukan PER METRIC, bukan berdasarkan posisi baris.
# Jadi satu toko dapat memiliki warna berbeda pada metric yang berbeda.
# ============================================================
for r in range(5, bottom_row + 1):
    # No + Kode
    for c in [2, 3]:
        ws.cell(r, c).alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    # Toko + Cabang
    for c in [4, 5]:
        ws.cell(r, c).alignment = Alignment(
            horizontal="left",
            vertical="center"
        )

    # JHK + seluruh angka/metric
    for c in range(6, 30):
        ws.cell(r, c).alignment = Alignment(
            horizontal="right",
            vertical="center"
        )

    # Semua cell data wajib mempunyai border
    for c in range(2, 30):
        ws.cell(r, c).border = border

# ------------------------------------------------------------
# TOP 5 / BOTTOM 5 PER METRIC
# ------------------------------------------------------------
# Mapping kolom Excel pada Resume Daily:
# G = SPD
# H = SPD TAG I
# O = % Ontime
# P = % Late
# ------------------------------------------------------------
metric_column_map = {
    "SPD": "G",
    "SPD TAG I": "H",
    "% Ontime": "O",
    "% Late": "P"
}

# Data yang ditampilkan pada Resume Daily sudah berada di final_report.
# Gunakan final_report sebagai sumber ranking agar posisi Excel tepat.
rank_data = final_report.copy().reset_index(drop=True)

for metric, excel_col in metric_column_map.items():
    values = pd.to_numeric(
        rank_data[
            {
                "SPD": "Daily SPD",
                "SPD TAG I": "Daily SPD TAG I",
                "% Ontime": "Daily % Ontime",
                "% Late": "Daily % Late"
            }[metric]
        ],
        errors="coerce"
    )

    # Posisi 0-based pada final_report -> Excel row 5 + posisi.
    valid = values.dropna()

    # Ambil 5 nilai terbesar dan 5 nilai terkecil.
    top_positions = valid.nlargest(5).index.tolist()
    bottom_positions = valid.nsmallest(5).index.tolist()

    if metric == "% Late":
        # Late dibalik: nilai paling kecil = performa terbaik (HIJAU).
        green_positions = bottom_positions
        red_positions = top_positions
    else:
        # SPD, SPD TAG I, Ontime: nilai terbesar = terbaik (HIJAU).
        green_positions = top_positions
        red_positions = bottom_positions

    for pos in green_positions:
        excel_row = 5 + int(pos)
        ws[f"{excel_col}{excel_row}"].fill = PatternFill(
            "solid",
            fgColor=GREEN
        )

    for pos in red_positions:
        excel_row = 5 + int(pos)
        ws[f"{excel_col}{excel_row}"].fill = PatternFill(
            "solid",
            fgColor=RED
        )

# ------------------------------------------------------------
# TOP 5 / BOTTOM 5 MTD PER METRIC
# ------------------------------------------------------------
# Resume Daily juga mempunyai blok MTD pada kolom Q:AC.
# Highlight MTD dibuat TERPISAH dari Daily karena ranking MTD
# harus berdasarkan nilai MTD masing-masing store.
#
# Mapping kolom Excel MTD:
# T  = SPD
# U  = SPD TAG I
# AB = % Ontime
# AC = % Late
#
# SPD / SPD TAG I / Ontime:
#   Top 5    = HIJAU
#   Bottom 5 = MERAH
#
# Late dibalik:
#   5 Late TERENDAH = HIJAU
#   5 Late TERTINGGI = MERAH
# ------------------------------------------------------------
mtd_metric_column_map = {
    "SPD": "T",
    "SPD TAG I": "U",
    "% Ontime": "AB",
    "% Late": "AC"
}

mtd_rank_source = final_report.copy().reset_index(drop=True)

mtd_source_columns = {
    "SPD": "MTD SPD",
    "SPD TAG I": "MTD SPD TAG I",
    "% Ontime": "MTD % Ontime",
    "% Late": "MTD % Late"
}

for metric, excel_col in mtd_metric_column_map.items():
    values = pd.to_numeric(
        mtd_rank_source[mtd_source_columns[metric]],
        errors="coerce"
    )

    valid = values.dropna()
    top_positions = valid.nlargest(5).index.tolist()
    bottom_positions = valid.nsmallest(5).index.tolist()

    if metric == "% Late":
        # Late rendah = lebih baik
        green_positions = bottom_positions
        red_positions = top_positions
    else:
        # SPD, SPD TAG I, Ontime tinggi = lebih baik
        green_positions = top_positions
        red_positions = bottom_positions

    for pos in green_positions:
        excel_row = 5 + int(pos)
        ws[f"{excel_col}{excel_row}"].fill = PatternFill(
            "solid",
            fgColor=GREEN
        )

    for pos in red_positions:
        excel_row = 5 + int(pos)
        ws[f"{excel_col}{excel_row}"].fill = PatternFill(
            "solid",
            fgColor=RED
        )

# Baris Average/Sum tetap dibedakan sebagai total row
for c in range(2, 30):
    ws.cell(bottom_row, c).fill = PatternFill(
        "solid",
        fgColor=BLUE_LIGHT
    )
    ws.cell(bottom_row, c).alignment = Alignment(
        horizontal="right" if c >= 6 else "center",
        vertical="center",
        wrap_text=True
    )

# widths
daily_widths = {
    "B": 14,
    "C": 10,
    "D": 26,
    "E": 16,
    "F": 10,
    "G": 15,
    "H": 15,
    "I": 10,
    "J": 13,
    "K": 10,
    "L": 13,
    "M": 13,
    "N": 13,
    "O": 12,
    "P": 12,
    "Q": 10,
    "R": 18,
    "S": 20,
    "T": 15,
    "U": 15,
    "V": 10,
    "W": 13,
    "X": 12,
    "Y": 12,
    "Z": 12,
    "AA": 12,
    "AB": 12,
    "AC": 12
}

for col, width in daily_widths.items():
    ws.column_dimensions[col].width = width

ws.freeze_panes = "F5"
ws.row_dimensions[1].height = 22
ws.row_dimensions[2].height = 20
ws.row_dimensions[3].height = 24
ws.row_dimensions[4].height = 34
for r in range(5, bottom_row + 1):
    ws.row_dimensions[r].height = 20

**RESUME WEEKLY SHARE**

In [ ]:
ws = wb.create_sheet(
    sheet_names()["weekly"]
)

ws.sheet_view.showGridLines = False

ws["B1"] = (
    "Perfomance Analysis Dark Store SPD & % OTD"
)

ws["B2"] = (
    f"Tanggal {period_label()}"
)

ws["B1"].font = Font(
    bold=True,
    size=14
)

ws["B2"].font = Font(
    bold=True
)

headers = [
    "Kode Toko",
    "Toko",
    "Cabang",
    "% Ontime",
    " SPD ",
    "Kategori % Ontime",
    " Kategori SPD "
]

for i, h in enumerate(
    headers,
    start=2
):
    ws.cell(3, i).value = h
    ws.cell(3, i).fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws.cell(3, i).font = Font(
        bold=True
    )
    ws.cell(3, i).alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )
    ws.cell(3, i).border = border

category_fills = {
    "Top 25%": GREEN,
    "50-75%": YELLOW,
    "25-50%": ORANGE,
    "Bottom 25%": RED
}

for idx, row in analysis.iterrows():
    rr = 4 + idx

    values = [
        row["KD_STORE"],
        row["NAMA_STORE"],
        row["NAMA_BRANCH"],
        row["MTD % Ontime"],
        row["MTD SPD"],
        row["Kategori % Ontime"],
        row["Kategori SPD"]
    ]

    for c, value in enumerate(
        values,
        start=2
    ):
        ws.cell(rr, c).value = value
        ws.cell(rr, c).border = border
        ws.cell(rr, c).alignment = Alignment(
            vertical="center"
        )

    ws[f"E{rr}"].number_format = "0.00%"
    ws[f"F{rr}"].number_format = "#,##0"

    for c in [7, 8]:
        value = ws.cell(rr, c).value
        fill = category_fills.get(value)

        if fill:
            ws.cell(rr, c).fill = PatternFill(
                "solid",
                fgColor=fill
            )

        ws.cell(rr, c).alignment = Alignment(
            horizontal="center"
        )

# Quartile boxes
for cell, value in [
    ("J3", "Q"),
    ("K3", "Batas Kuartil OTD"),
    ("M3", "Q"),
    ("N3", "Batas Kuartil SPD")
]:
    ws[cell] = value
    ws[cell].fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws[cell].font = Font(
        bold=True
    )
    ws[cell].alignment = Alignment(
        horizontal="center"
    )
    ws[cell].border = border

for row_no, (q, label) in enumerate(
    [
        (0.25, "Q1"),
        (0.50, "Q2"),
        (0.75, "Q3")
    ],
    start=4
):
    ws[f"J{row_no}"] = label
    ws[f"K{row_no}"] = float(
        q_otd.loc[q]
    )
    ws[f"M{row_no}"] = label
    ws[f"N{row_no}"] = float(
        q_spd.loc[q]
    )

    for cell in [
        f"J{row_no}",
        f"K{row_no}",
        f"M{row_no}",
        f"N{row_no}"
    ]:
        ws[cell].border = border
        ws[cell].alignment = Alignment(
            horizontal="center"
        )

    ws[f"K{row_no}"].number_format = "0%"
    ws[f"N{row_no}"].number_format = "#,##0"

# 4 x 4 matrix
matrix_cols = [
    "Bottom 25%",
    "25-50%",
    "50-75%",
    "Top 25%"
]

matrix_rows = [
    "Bottom 25%",
    "25-50%",
    "50-75%",
    "Top 25%"
]

matrix_counts = pd.crosstab(
    pd.Categorical(
        analysis["Kategori SPD"],
        categories=matrix_rows,
        ordered=True
    ),
    pd.Categorical(
        analysis["Kategori % Ontime"],
        categories=matrix_cols,
        ordered=True
    )
).reindex(
    index=matrix_rows,
    columns=matrix_cols,
    fill_value=0
)

# Merge layout exactly like the output
ws.merge_cells("J8:K9")
ws.merge_cells("J10:J13")
ws.merge_cells("J14:K14")
ws.merge_cells("L8:O8")
ws.merge_cells("P8:P9")

# Seluruh area header matrix diberi warna, termasuk area kiri atas
# agar tidak ada cell header putih.
for r in range(8, 10):
    for c in range(10, 17):
        ws.cell(r, c).fill = PatternFill(
            "solid",
            fgColor=BLUE_HEADER
        )
        ws.cell(r, c).font = Font(bold=True)
        ws.cell(r, c).alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        ws.cell(r, c).border = border

ws["L8"] = "OTD"
ws["P8"] = "Total"

for cell in ["L8", "P8"]:
    ws[cell].fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws[cell].font = Font(
        bold=True
    )
    ws[cell].alignment = Alignment(
        horizontal="center"
    )
    ws[cell].border = border

for j, header in enumerate(
    matrix_cols,
    start=12
):
    ws.cell(9, j).value = header
    ws.cell(9, j).fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws.cell(9, j).font = Font(
        bold=True
    )
    ws.cell(9, j).alignment = Alignment(
        horizontal="center"
    )
    ws.cell(9, j).border = border

ws["J10"] = "SPD"

for i, row_cat in enumerate(
    matrix_rows,
    start=10
):
    ws.cell(i, 11).value = row_cat
    ws.cell(i, 11).fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws.cell(i, 11).font = Font(
        bold=True
    )
    ws.cell(i, 11).border = border
    ws.cell(i, 11).alignment = Alignment(
        horizontal="center"
    )

    for j, col_cat in enumerate(
        matrix_cols,
        start=12
    ):
        ws.cell(i, j).value = int(
            matrix_counts.loc[
                row_cat,
                col_cat
            ]
        )
        ws.cell(i, j).border = border
        ws.cell(i, j).alignment = Alignment(
            horizontal="center"
        )

    ws.cell(i, 16).value = int(
        matrix_counts.loc[row_cat].sum()
    )
    ws.cell(i, 16).border = border
    ws.cell(i, 16).alignment = Alignment(
        horizontal="center"
    )

ws["J14"] = "Total"
ws["J14"].font = Font(
    bold=True
)
ws["J14"].fill = PatternFill(
    "solid",
    fgColor=BLUE_HEADER
)
ws["J14"].border = border

for j, col_cat in enumerate(
    matrix_cols,
    start=12
):
    ws.cell(14, j).value = int(
        matrix_counts[col_cat].sum()
    )
    ws.cell(14, j).border = border
    ws.cell(14, j).font = Font(
        bold=True
    )
    ws.cell(14, j).alignment = Alignment(
        horizontal="center"
    )

ws["P14"] = int(
    matrix_counts.values.sum()
)
ws["P14"].border = border
ws["P14"].font = Font(
    bold=True
)

# Keterangan
ws.merge_cells("J17:P22")

ws["J17"] = (
    "Keterangan:\n"
    "• Kategori SPD dan %OTD ditentukan berdasarkan quartile "
    "(Q1, Q2, dan Q3) dari masing-masing metrik.\n"
    "• Bottom 25% merupakan nilai < Q1.\n"
    "• 25–50% merupakan nilai ≥ Q1 dan < Q2.\n"
    "• 50–75% merupakan nilai ≥ Q2 dan < Q3.\n"
    "• Top 25% merupakan nilai ≥ Q3."
)

ws["J17"].alignment = Alignment(
    wrap_text=True,
    vertical="top"
)

ws["J17"].border = Border(
    left=Side(style="medium"),
    right=Side(style="medium"),
    top=Side(style="medium"),
    bottom=Side(style="medium")
)

weekly_widths = {
    "B": 12,
    "C": 25,
    "D": 16,
    "E": 13,
    "F": 15,
    "G": 18,
    "H": 18,
    "J": 8,
    "K": 15,
    "L": 15,
    "M": 15,
    "N": 15,
    "O": 15,
    "P": 10
}

for col, width in weekly_widths.items():
    ws.column_dimensions[col].width = width

# ============================================================
# FINAL FORMATTING SHEET 2 — PERFORMANCE ANALYSIS
# ============================================================
# Tabel utama benar-benar berkotak, alignment konsisten, dan
# tidak ada area data yang dibiarkan tanpa border.
for r in range(3, 4 + len(analysis)):
    for c in range(2, 9):
        cell = ws.cell(r, c)
        cell.border = border
        if r >= 4:
            if c in [2]:
                cell.alignment = Alignment(horizontal="center", vertical="center")
            elif c in [3, 4]:
                cell.alignment = Alignment(horizontal="left", vertical="center")
            elif c in [5, 6]:
                cell.alignment = Alignment(horizontal="right", vertical="center")
            else:
                cell.alignment = Alignment(horizontal="center", vertical="center")

# Quartile boxes dibuat konsisten dan berkotak
for r in range(3, 7):
    for c in [10, 11, 13, 14]:
        ws.cell(r, c).border = border
        ws.cell(r, c).alignment = Alignment(horizontal="center", vertical="center")

# Matrix dan keterangan diberi border pada seluruh area yang dipakai
for r in range(8, 15):
    for c in range(10, 17):
        ws.cell(r, c).border = border
        ws.cell(r, c).alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

for r in range(17, 23):
    for c in range(10, 17):
        ws.cell(r, c).border = Border(
            left=Side(style="medium", color="7F7F7F"),
            right=Side(style="medium", color="7F7F7F"),
            top=Side(style="medium", color="7F7F7F"),
            bottom=Side(style="medium", color="7F7F7F")
        )

ws.freeze_panes = "B4"
ws.row_dimensions[1].height = 22
ws.row_dimensions[2].height = 20
ws.row_dimensions[3].height = 30
for r in range(4, 4 + len(analysis)):
    ws.row_dimensions[r].height = 20

**SHEET PENDUKUNG SHARE**

In [ ]:
# SHEET 3–10
# ============================================================
sheet_data = {
    "Pivot DAILY": pivot_daily,
    "Pivot WEEKLY": pivot_weekly,

    # Data transaksi = bahan transaksi setelah filter DARKSTORE
    "Data Transaksi": tx.drop(
        columns=["DATE"],
        errors="ignore"
    ),

    sheet_names()["oos_daily"]: oos_daily,
    sheet_names()["oos_mtd"]: oos_mtd,

    "SLA REMARK": sla.drop(
        columns=["DATE"],
        errors="ignore"
    ),

    sheet_names()["sla_daily"]: sla_19,
    sheet_names()["sla_mtd"]: sla_mtd_out
}

for sheet_name, df in sheet_data.items():

    ws = wb.create_sheet(sheet_name)
    ws.sheet_view.showGridLines = False

    # Header
    for c, header in enumerate(
        df.columns,
        start=1
    ):
        cell = ws.cell(1, c)
        cell.value = header
        cell.fill = PatternFill(
            "solid",
            fgColor=BLUE_HEADER
        )
        cell.font = Font(
            bold=True
        )
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        cell.border = border

    # Data
    for r_idx, row in enumerate(
        df.itertuples(
            index=False,
            name=None
        ),
        start=2
    ):
        for c_idx, value in enumerate(
            row,
            start=1
        ):
            ws.cell(
                r_idx,
                c_idx
            ).value = value
            ws.cell(
                r_idx,
                c_idx
            ).border = border

    ws.freeze_panes = "A2"

    auto_width(ws)

# Percentage formats
for r in range(
    2,
    wb["Pivot DAILY"].max_row + 1
):
    for col in [
        "I", "J", "K", "L", "M", "N"
    ]:
        wb["Pivot DAILY"][
            f"{col}{r}"
        ].number_format = "0.00%"

for r in range(
    2,
    wb["Pivot WEEKLY"].max_row + 1
):
    for col in [
        "I", "J", "K", "L", "M", "N"
    ]:
        wb["Pivot WEEKLY"][
            f"{col}{r}"
        ].number_format = "0.00%"

for sheet_name in [
    sheet_names()["sla_daily"],
    sheet_names()["sla_mtd"]
]:
    ws = wb[sheet_name]

    for r in range(
        2,
        ws.max_row + 1
    ):
        if ws.max_column >= 5:
            ws[f"E{r}"].number_format = "0.00%"

        if ws.max_column >= 6:
            ws[f"F{r}"].number_format = "0.00%"

# OOS format
for sheet_name in [
    sheet_names()["oos_daily"],
    sheet_names()["oos_mtd"]
]:
    ws = wb[sheet_name]

    for r in range(
        2,
        ws.max_row + 1
    ):
        for col in ["D", "E", "F", "G"]:
            ws[
                f"{col}{r}"
            ].number_format = "0.00%"


**FORMATTING HIGHLIGHT DAN BORDER WEEKLY DAILY RESUME SHARE**

In [ ]:
# FINAL RE-APPLY TOP 5 / BOTTOM 5 HIGHLIGHT
# ============================================================
# Diterapkan PALING AKHIR sebelum workbook disimpan agar warna
# tidak tertimpa oleh formatting sheet lain.
# Ranking berdasarkan KODE TOKO, bukan nomor/index dataframe.
#
# DAILY:
#   G = SPD          -> Top 5 hijau, Bottom 5 merah
#   H = SPD TAG I    -> Top 5 hijau, Bottom 5 merah
#   O = % Ontime     -> Top 5 hijau, Bottom 5 merah
#   P = % Late       -> 5 terendah hijau, 5 tertinggi merah
#
# MTD 1-tanggal maksimum:
#   T = SPD          -> Top 5 hijau, Bottom 5 merah
#   U = SPD TAG I    -> Top 5 hijau, Bottom 5 merah
#   AB = % Ontime    -> Top 5 hijau, Bottom 5 merah
#   AC = % Late      -> 5 terendah hijau, 5 tertinggi merah
# ============================================================
ws = wb[sheet_names()["daily"]]

# Ambil baris Excel berdasarkan Kode Toko supaya tidak bergantung
# pada index dataframe.
excel_row_by_store = {}
for r in range(5, bottom_row):
    code = ws[f"C{r}"].value
    if code is not None:
        excel_row_by_store[str(code).strip()] = r

# Data ranking Daily dan MTD berdasarkan Kode Toko.
rank_daily = final_report.copy()
rank_mtd = final_report.copy()

# Mapping metric -> kolom sumber dataframe -> kolom Excel
highlight_maps = [
    (rank_daily, {
        "SPD": ("Daily SPD", "G"),
        "SPD TAG I": ("Daily SPD TAG I", "H"),
        "% Ontime": ("Daily % Ontime", "O"),
        "% Late": ("Daily % Late", "P")
    }),
    (rank_mtd, {
        "SPD": ("MTD SPD", "T"),
        "SPD TAG I": ("MTD SPD TAG I", "U"),
        "% Ontime": ("MTD % Ontime", "AB"),
        "% Late": ("MTD % Late", "AC")
    })
]

for source_df, metric_map in highlight_maps:
    source_df = source_df.copy()
    source_df["__CODE__"] = source_df["Kode"].astype(str).str.strip()

    for metric, (source_col, excel_col) in metric_map.items():
        values = pd.to_numeric(
            source_df[source_col],
            errors="coerce"
        )

        valid_df = pd.DataFrame({
            "code": source_df["__CODE__"],
            "value": values
        }).dropna(subset=["value"])

        # Hilangkan warna lama pada metric ini terlebih dahulu.
        for code in valid_df["code"]:
            if code in excel_row_by_store:
                ws[f"{excel_col}{excel_row_by_store[code]}"].fill = PatternFill(
                    fill_type=None
                )

        # Top/bottom berdasarkan nilai metric.
        top_codes = valid_df.nlargest(5, "value")["code"].tolist()
        bottom_codes = valid_df.nsmallest(5, "value")["code"].tolist()

        if metric == "% Late":
            # Late terendah = performa terbaik = hijau.
            green_codes = bottom_codes
            red_codes = top_codes
        else:
            # SPD, SPD TAG I, Ontime tertinggi = performa terbaik = hijau.
            green_codes = top_codes
            red_codes = bottom_codes

        for code in green_codes:
            if code in excel_row_by_store:
                ws[f"{excel_col}{excel_row_by_store[code]}"].fill = PatternFill(
                    "solid",
                    fgColor=GREEN
                )

        for code in red_codes:
            if code in excel_row_by_store:
                ws[f"{excel_col}{excel_row_by_store[code]}"].fill = PatternFill(
                    "solid",
                    fgColor=RED
                )

# Pastikan border tetap ada setelah fill.
for r in range(5, bottom_row):
    for c in range(2, 30):
        ws.cell(r, c).border = border

# Pastikan header MTD tetap berwarna setelah semua formatting.
for c in range(17, 30):  # Q:AC
    ws.cell(3, c).fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )
    ws.cell(4, c).fill = PatternFill(
        "solid",
        fgColor=BLUE_HEADER
    )

# ============================================================
# FINAL GLOBAL BORDER + HEADER STANDARDIZATION
# ============================================================
# Semua header utama menggunakan warna yang SAMA dan seluruh
# cell yang dipakai diberi border agar tidak ada area putih/
# tanpa kotak seperti pada output sebelumnya.
# ============================================================
HEADER_FILL = PatternFill(
    fill_type="solid",
    fgColor=BLUE_HEADER
)

# SHEET 1 — seluruh area tabel B:AC
ws = wb[sheet_names()["daily"]]

# Header merged + subheader: semua warna sama
for r in [3, 4]:
    for c in range(2, 30):
        cell = ws.cell(r, c)
        cell.fill = HEADER_FILL
        cell.font = Font(bold=True)
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        cell.border = border

# Seluruh tabel Daily wajib berkotak
for r in range(3, bottom_row + 1):
    for c in range(2, 30):
        ws.cell(r, c).border = border

# Header merged perlu border di seluruh area fisiknya
for r1, r2, c1, c2 in [
    (3, 4, 2, 2),
    (3, 4, 3, 3),
    (3, 4, 4, 4),
    (3, 4, 5, 5),
    (3, 3, 6, 16),
    (3, 3, 17, 29),
]:
    for r in range(r1, r2 + 1):
        for c in range(c1, c2 + 1):
            ws.cell(r, c).fill = HEADER_FILL
            ws.cell(r, c).border = border

# SHEET 2 — Performance Analysis
ws = wb[sheet_names()["weekly"]]

# Header tabel utama B:H
for r in range(3, 4):
    for c in range(2, 9):
        ws.cell(r, c).fill = HEADER_FILL
        ws.cell(r, c).font = Font(bold=True)
        ws.cell(r, c).alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        ws.cell(r, c).border = border

# Quartile header J:K dan M:N
for c in [10, 11, 13, 14]:
    ws.cell(3, c).fill = HEADER_FILL
    ws.cell(3, c).font = Font(bold=True)
    ws.cell(3, c).alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )
    ws.cell(3, c).border = border

# Matrix seluruh area J8:P14 berkotak
for r in range(8, 15):
    for c in range(10, 17):
        ws.cell(r, c).border = border

# Matrix header — semuanya warna header yang sama
for r in [8, 9]:
    for c in range(10, 17):
        ws.cell(r, c).fill = HEADER_FILL
        ws.cell(r, c).font = Font(bold=True)
        ws.cell(r, c).alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )
        ws.cell(r, c).border = border

# Matrix row labels + total tetap memiliki border
for r in range(10, 15):
    for c in range(10, 17):
        ws.cell(r, c).border = border
        if c in [10, 11] or r == 14:
            ws.cell(r, c).fill = HEADER_FILL
            ws.cell(r, c).font = Font(bold=True)

# Keterangan box tetap berkotak
for r in range(17, 23):
    for c in range(10, 17):
        ws.cell(r, c).border = border

# Semua cell tabel utama Analysis juga berkotak
for r in range(3, 4 + len(analysis)):
    for c in range(2, 9):
        ws.cell(r, c).border = border

# Semua quartile data cell berkotak
for r in range(3, 7):
    for c in [10, 11, 13, 14]:
        ws.cell(r, c).border = border

# Save
wb.save(OUTPUT_FILE)

# ============================================================
# FINAL VALIDATION
# ============================================================
check = load_workbook(
    OUTPUT_FILE,
    data_only=False
)

expected_sheets = [
    sheet_names()["daily"],
    sheet_names()["weekly"],
    "Pivot DAILY",
    "Pivot WEEKLY",
    "Data Transaksi",
    sheet_names()["oos_daily"],
    sheet_names()["oos_mtd"],
    "SLA REMARK",
    sheet_names()["sla_daily"],
    sheet_names()["sla_mtd"]
]

missing_sheets = [
    s for s in expected_sheets
    if s not in check.sheetnames
]

if missing_sheets:
    raise ValueError(
        f"Sheet output kurang: {missing_sheets}"
    )

print("=" * 70)
print("SELESAI")
print("=" * 70)
print("Output :", OUTPUT_FILE)
print("Store  :", len(final_report))
print("Q1/Q2/Q3 SPD :", q_spd.to_dict())
print("Q1/Q2/Q3 OTD :", q_otd.to_dict())
print("=" * 70)

files.download(OUTPUT_FILE)


SELESAI
Output : Resume MTD DS 29 SEPTEMBER 2026.xlsx
Store  : 68
Q1/Q2/Q3 SPD : {0.25: 39798292.17899938, 0.5: 45988930.16083744, 0.75: 55924031.58810345}
Q1/Q2/Q3 OTD : {0.25: 0.678742857881937, 0.5: 0.7944301577917333, 0.75: 0.8889652161862527}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>